# DS2002 · Midterm Checkpoint

**Lab — 2026-10-23 · Fall 2026**  

---

## Lab 09 — Midterm checkpoint, week 1 of 3

**What you submit: the link to your team's main repo. Only that.** One submission per team. Do not upload separate files.

Put this notebook in the repo (the `notebooks/` folder is fine) and push it, so everything we check is in one place.

**You do not have to fill this all out.** It is a checkpoint, not a finished project. Skip any section you have not started and write "not started" next to it. What we must see is:

1. **Your charter**, committed to the repo
2. **Any progress** you have made, with code that ran and output you can point at

Start with the team setup, then work down the same eight things we listed at Monday's kickoff, as far as you have gotten. This is graded on evidence, not on intentions.

The purpose is to catch a stalled team while there is still time to unstall it. If something is not working, say so here plainly. A checkpoint that honestly reports "the API join is not working and here is the error" is worth more than one that claims progress that does not exist, and it gets you help on Monday.

### Team setup

This part is required. Fill it in once for the whole team.

You need:

- Your team name, and every member with a GitHub handle (2 to 4 people)
- A link to your team's **main repo**. This is the one link you submit.
- A link to `CHARTER.md` in that repo, the page on GitHub, not a file on someone's laptop

Nice to have, and also in your charter: this week's notebook owner, your meeting time, and the five worst problems in the data from the profile you ran on Monday. Be specific: "`unit_price` is stored as text and some values start with `$`" beats "prices are messy."

In [1]:
team_setup = {
    'team_name': 'DS 2002 Group',
    'members': [
        {'name': 'Ciara Graves', 'github': 'ciara-graves'},
        {'name': 'Kiersten Bruns', 'github': 'kierstenbruns-lab'},
        {'name': 'Camden Lannin', 'github': 'lannincamden'},
        {'name': 'Olympia Sauerwein', 'github': 'olympiasau'},
    ],
    'repo_url': 'https://github.com/ciara-graves/walmart-hurricane-project',
    'charter_url': 'https://github.com/ciara-graves/walmart-hurricane-project/blob/main/CHARTER.md',
    'notebook_owner_this_week': 'Ciara Graves',
    'meeting_time': 'M/W/F at 12:00',
    'five_worst_problems': [
        'Store IDs come in inconsistent formats (FL-287, fl_287, FL287)',
        'The same product is sold under multiple SKUs and name spellings',
        'Timestamps appear in five different formats, and some are missing',
        'Negative quantities (returns) and duplicate transactions inflate or distort revenue',
        'Missing values are written many ways (NULL, N/A, -, blank)',
    ],
}

def check_team(t, quiet=False):
    def real(v):
        return isinstance(v, str) and v.strip() != '' and not v.startswith('TODO')
    required = {
        'team name': real(t['team_name']),
        '2 to 4 members, each with a name and GitHub handle':
            2 <= len(t['members']) <= 4 and all(real(m['name']) and real(m['github']) for m in t['members']),
        'main repo link': real(t['repo_url']) and t['repo_url'].startswith('https://github.com/'),
        'charter link points at CHARTER.md': real(t['charter_url']) and t['charter_url'].endswith('CHARTER.md'),
    }
    nice = {
        'notebook owner': real(t['notebook_owner_this_week']),
        'meeting time': real(t['meeting_time']),
        'five specific problems': len(t['five_worst_problems']) >= 5 and all(real(p) for p in t['five_worst_problems']),
    }
    if not quiet:
        print('REQUIRED')
        for label, good in required.items():
            print('  PASS' if good else '  FAIL', '-', label)
        print('NICE TO HAVE')
        for label, good in nice.items():
            print('  done' if good else '  not yet', '-', label)
        print()
        print('team setup complete:', all(required.values()))
    return all(required.values())

check_team(team_setup)

REQUIRED
  PASS - team name
  PASS - 2 to 4 members, each with a name and GitHub handle
  PASS - main repo link
  PASS - charter link points at CHARTER.md
NICE TO HAVE
  done - notebook owner
  done - meeting time
  done - five specific problems

team setup complete: True


True

Now show that the charter is actually committed. This is required. Run the next cell and leave its output in the notebook. Point `REPO_PATH` at your local clone of the team repo. If it prints a line like `a1b2c3d Add team charter`, you are done. If it prints "no commits found," the charter is not in the repo yet, so commit it and push before you submit.

In [2]:
import subprocess

REPO_PATH = '../walmart-hurricane-project'      # your local clone of the team repo

result = subprocess.run(['git', '-C', REPO_PATH, 'log', '--oneline', '--', 'CHARTER.md'],
                        capture_output=True, text=True)
print(result.stdout.strip() or 'no commits found for CHARTER.md')

no commits found for CHARTER.md


### 1 — Your cleaning pipeline as functions

**TODO:** paste the cleaning code you have written so far. It should be functions, not loose cells, and it should cover at minimum: duplicates, `unit_price` to numeric (it comes in as text, some with a `$`), the negative `quantity` values with your decision applied, `timestamp` parsed, and `store_id` cleaned up so it will join to the store file.

Print the row count before and after. We want to see how many rows your decisions removed.

In [3]:
!git clone https://github.com/ciara-graves/walmart-hurricane-project.git
%cd walmart-hurricane-project/data
!ls -lh

fatal: destination path 'walmart-hurricane-project' already exists and is not an empty directory.
/content/walmart-hurricane-project/data
total 2.4M
-rw-r--r-- 1 root root  439 Oct 26 09:14 distribution_centers.csv
-rw-r--r-- 1 root root    1 Oct 26 09:14 Initial
-rw-r--r-- 1 root root 344K Oct 26 09:14 inventory_and_sales.db
-rw-r--r-- 1 root root 3.1K Oct 26 09:14 product_catalog.csv
-rw-r--r-- 1 root root 1.2K Oct 26 09:14 store_locations.csv
-rw-r--r-- 1 root root 2.1M Oct 26 09:14 walmart_transactions.csv


In [4]:
import re
import sqlite3
import numpy as np
import pandas as pd

DATA = "/content/walmart-hurricane-project/data"

MISSING_TOKENS = {"", "NULL", "N/A", "NA", "NAN", "NONE", "-"}

CATEGORY_MAP = {
    "emergency supplies": "Emergency", "groceries": "Grocery",
    "games & entertainment": "Entertainment", "household essentials": "Household",
    "baby products": "Baby", "breakfast foods": "Breakfast",
    "alcoholic beverages": "Alcohol", "health & wellness": "Health",
}

PRODUCT_SKUS = {
    "Pop-Tarts Strawberry": ["PT-12", "SKU#459812", "POPTART-STRAW", "PT_STRAWBERRY"],
    "Bottled Water 24-Pack": ["WTR-24", "WATER24PK", "WTR_24PK"],
    "AA Batteries 4-Pack": ["BATT-100", "BAT-AA4", "BATT_AA_4PK"],
    "Monopoly Board Game": ["BG-MONO", "MONOPOLY-01", "GAME-MONO"],
    "LED Flashlight": ["FLASH-01", "LED-FLASH"],
    "Light Beer 24-Pack": ["BEER-24", "LTBEER24"],
    "Playing Cards": ["CARDS-01", "PLAYCARD-STD"],
    "Roku Streaming Stick": ["STREAM-01", "ROKU-STD"],
    "Canned Soup": ["SOUP-CAN", "CNSOUP-01"],
    "White Bread": ["BREAD-WH", "WHITEBREAD"],
    "Peanut Butter 16oz": ["PB-16", "PNUTBTR-16"],
    "Emergency Candles": ["CNDL-EM", "CANDLE-EMRG"],
    "First Aid Kit": ["FAID-01", "FIRSTAID-KIT"],
    "Portable Phone Charger": ["CHRG-PORT", "PORTCHARGER"],
    "Chips Variety Pack": ["CHIPS-VP", "CHIPVARIETY"],
    "Paper Towels 6-Roll": ["PTOWEL-6", "PAPERTOWEL6"],
    "Toilet Paper 12-Roll": ["TP-12", "TOILETPAPER12"],
    "Diapers Size 3": ["DIAP-S3", "DIAPERS-3"],
    "Infant Formula": ["BFORM-01", "BABYFORMULA"],
    "Heavy Duty Tarp": ["TARP-HD", "HDTARP-01"],
}
SKU_TO_PRODUCT = {s: p for p, skus in PRODUCT_SKUS.items() for s in skus}

CENTRAL_STORES = {"FL-710"}

TS_FORMATS = ["%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S", "%m/%d/%Y %H:%M",
              "%d-%b-%Y %H:%M", "%m-%d-%Y %H:%M:%S"]


def standardize_missing(df):
    out = df.copy()
    for c in out.columns:
        if out[c].dtype == object or str(out[c].dtype) == "str":
            s = out[c].astype("string").str.strip()
            out[c] = s.mask(s.str.upper().isin(MISSING_TOKENS))
    return out


def clean_store_id(s):
    digits = s.astype("string").str.extract(r"(\d{3})")[0]
    return ("FL-" + digits).where(digits.notna())


def clean_category(s):
    low = s.astype("string").str.strip().str.lower()
    mapped = low.map(CATEGORY_MAP)
    return mapped.fillna(low.str.title())


def to_numeric_money(s):
    return pd.to_numeric(s.astype("string").str.replace(r"[$,\s]", "", regex=True), errors="coerce")


def parse_timestamp(s):
    s = s.astype("string").str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    for fmt in TS_FORMATS:
        todo = out.isna() & s.notna()
        if not todo.any():
            break
        out = out.fillna(pd.to_datetime(s.where(todo), format=fmt, errors="coerce"))
    return out


def clean_text_and_ids(df):
    out = df.copy()
    out["store_id"] = clean_store_id(out["store_id"])
    out["category"] = clean_category(out["category"])
    out["sku"] = out["sku"].astype("string").str.strip().str.upper()
    out["product_name"] = out["product_name"].astype("string").str.strip()
    return out


def clean_numbers(df):
    out = df.copy()
    out["unit_price"] = to_numeric_money(out["unit_price"])
    out["quantity"] = pd.to_numeric(out["quantity"], errors="coerce")
    return out


def clean_timestamps(df):
    out = df.copy()
    out["timestamp"] = parse_timestamp(out["timestamp"])
    out["has_timestamp"] = out["timestamp"].notna()
    return out


def handle_negative_quantity(df):
    out = df.copy()
    out["is_return"] = out["quantity"] < 0
    out["revenue"] = (out["quantity"] * out["unit_price"]).round(2)
    return out


def consolidate_products(df):
    out = df.copy()
    out["canonical_product"] = out["sku"].map(SKU_TO_PRODUCT)
    return out


def drop_duplicates(df):
    content = [c for c in df.columns if c != "transaction_id"]
    df = df.sort_values("transaction_id", kind="stable")
    out = df.drop_duplicates(keep="first")
    n1 = len(df) - len(out)
    has_ts = out["timestamp"].notna()
    repeat = out.duplicated(subset=content, keep="first") & has_ts
    out = out[~repeat]
    print(f"  duplicates: {n1} exact + {int(repeat.sum())} repeated under a new transaction_id")
    return out


def clean_transactions(raw):
    n0 = len(raw)
    out = (raw.pipe(standardize_missing).pipe(clean_text_and_ids).pipe(clean_numbers)
              .pipe(clean_timestamps).pipe(handle_negative_quantity)
              .pipe(consolidate_products).pipe(drop_duplicates).reset_index(drop=True))
    print(f"transactions  rows before: {n0:,}   rows after: {len(out):,}   removed: {n0-len(out):,}")
    print(f"  kept but flagged: {int(out.is_return.sum())} returns, "
          f"{int((~out.has_timestamp).sum())} rows with no timestamp")
    return out


def clean_stores(raw):
    n0 = len(raw)
    out = standardize_missing(raw)
    out["store_id"] = clean_store_id(out["store_id"])
    out["state"] = "FL"
    out["latitude"] = pd.to_numeric(out["latitude"], errors="coerce")
    out["longitude"] = pd.to_numeric(out["longitude"], errors="coerce")
    out["timezone"] = np.where(out["store_id"].isin(CENTRAL_STORES), "America/Chicago", "America/New_York")
    out = (out.assign(_n=out.notna().sum(axis=1)).sort_values("_n", ascending=False)
              .drop_duplicates("store_id").drop(columns="_n").sort_values("store_id").reset_index(drop=True))
    print(f"stores        rows before: {n0}   rows after: {len(out)}   "
          f"(missing coordinates: {out.latitude.isna().sum()})")
    return out


def clean_catalog(raw):
    n0 = len(raw)
    out = standardize_missing(raw)
    out["sku"] = out["sku"].str.upper()
    out["unit_cost"] = pd.to_numeric(out["unit_cost"], errors="coerce")
    out["category"] = clean_category(out["category"])
    out["canonical_product"] = out["sku"].map(SKU_TO_PRODUCT)
    print(f"catalog       rows before: {n0}   rows after: {len(out)}   "
          f"unmapped skus: {int(out.canonical_product.isna().sum())}   missing unit_cost: {int(out.unit_cost.isna().sum())}")
    return out


def clean_distribution_centers(raw):
    n0 = len(raw)
    out = standardize_missing(raw)
    out["surge_capacity_is_approx"] = out["surge_capacity"].str.contains("~", na=False)
    out["surge_capacity"] = pd.to_numeric(out["surge_capacity"].str.replace("~", "", regex=False), errors="coerce")
    for c in ["avg_daily_shipments", "distance_to_orlando_mi", "cost_per_shipment"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    print(f"dist. centers rows before: {n0}   rows after: {len(out)}   "
          f"(missing cost_per_shipment: {int(out.cost_per_shipment.isna().sum())})")
    return out


def clean_daily_sales_summary(raw):
    n0 = len(raw)
    out = raw.copy()
    out["date"] = pd.to_datetime(out["date"].astype(str).str.strip().str.replace("-", ""), format="%Y%m%d")
    out["store_id"] = clean_store_id(out["store_id"])
    out["category"] = clean_category(out["category"])
    out["has_net_return_day"] = out["total_revenue"] < 0
    out = out.drop_duplicates(["date", "store_id", "category"])
    print(f"daily_sales   rows before: {n0:,}   rows after: {len(out):,}   "
          f"(negative-revenue rows kept & flagged: {int(out.has_net_return_day.sum())})")
    return out


def clean_inventory(raw):
    n0 = len(raw)
    out = raw.copy()
    out["date"] = pd.to_datetime(out["date"])
    out["store_id"] = clean_store_id(out["store_id"])
    out["sku"] = out["sku"].str.strip().str.upper()
    out["canonical_product"] = out["sku"].map(SKU_TO_PRODUCT)
    out["sku_in_catalog"] = out["canonical_product"].notna()
    out = out.drop_duplicates(["date", "store_id", "sku"])
    print(f"inventory     rows before: {n0:,}   rows after: {len(out):,}   "
          f"(rows with sku not in catalog, kept & flagged: {int((~out.sku_in_catalog).sum())}; "
          f"null units_on_order left as NaN: {int(out.units_on_order.isna().sum())})")
    return out


def load_and_clean_all(data_dir=DATA):
    rd = lambda f: pd.read_csv(f"{data_dir}/{f}", dtype=str, keep_default_na=False)
    con = sqlite3.connect(f"{data_dir}/inventory_and_sales.db")
    tables = dict(
        transactions=clean_transactions(rd("walmart_transactions.csv")),
        stores=clean_stores(rd("store_locations.csv")),
        catalog=clean_catalog(rd("product_catalog.csv")),
        dcs=clean_distribution_centers(rd("distribution_centers.csv")),
        daily_sales=clean_daily_sales_summary(pd.read_sql("select * from daily_sales_summary", con)),
        inventory=clean_inventory(pd.read_sql("select * from inventory_levels", con)),
    )
    con.close()
    return tables


tables = load_and_clean_all()

  duplicates: 30 exact + 30 repeated under a new transaction_id
transactions  rows before: 25,562   rows after: 25,502   removed: 60
  kept but flagged: 130 returns, 884 rows with no timestamp
stores        rows before: 14   rows after: 12   (missing coordinates: 1)
catalog       rows before: 45   rows after: 45   unmapped skus: 0   missing unit_cost: 5
dist. centers rows before: 5   rows after: 5   (missing cost_per_shipment: 1)
daily_sales   rows before: 3,600   rows after: 3,600   (negative-revenue rows kept & flagged: 34)
inventory     rows before: 4,320   rows after: 4,320   (rows with sku not in catalog, kept & flagged: 355; null units_on_order left as NaN: 626)


### 2 — Your decision log

**TODO:** every cleaning decision, what it did, and how many rows it touched. If your teammates disagree with one of these, note the disagreement — that is useful information, not a problem to hide.

In [5]:
raw_tx = pd.read_csv(f"{DATA}/walmart_transactions.csv", dtype=str, keep_default_na=False)
raw_st = pd.read_csv(f"{DATA}/store_locations.csv", dtype=str, keep_default_na=False)
raw_dc = pd.read_csv(f"{DATA}/distribution_centers.csv", dtype=str, keep_default_na=False)

tx = tables["transactions"]

def n_changed(before, after):
    b = before.fillna("").astype(str).str.strip()
    a = after.fillna("").astype(str)
    return int((b != a).sum())

missing_cells = int(sum(raw_tx[c].str.strip().str.upper().isin(MISSING_TOKENS).sum() for c in raw_tx.columns))
std_ts = raw_tx["timestamp"].str.strip().str.match(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$")
nonstd_ts = int((~std_ts & (raw_tx["timestamp"].str.strip() != "")).sum())

log = [
    ("transactions",
     "Made all the different ways of writing 'missing' (blank, NULL, N/A, NA, NaN, None, -) into one real missing value.",
     "Python only treats real missing values as missing. Without this, 'N/A' would be counted as actual data.",
     missing_cells),
    ("transactions",
     "Made every store ID look the same: FL-###.",
     "The same store was written many ways (FL-287, fl_287, FL287). They have to match to join to the store file.",
     n_changed(raw_tx["store_id"], clean_store_id(raw_tx["store_id"]))),
    ("transactions",
     "Changed category names to the same 10 names used in the daily sales summary.",
     "So our transaction totals can be compared to the summary table.",
     n_changed(raw_tx["category"], clean_category(raw_tx["category"]))),
    ("transactions",
     "Removed extra spaces and made every SKU uppercase.",
     "'pt-12' and 'PT-12 ' would otherwise look like two different products.",
     n_changed(raw_tx["sku"], raw_tx["sku"].str.strip().str.upper())),
    ("transactions",
     "Turned unit_price from text into a number by removing $, commas, and spaces.",
     "Prices came in as text, and some had a $, so we could not do math on them.",
     int(raw_tx["unit_price"].str.contains(r"[$,\s]", na=False).sum())),
    ("transactions",
     "Turned timestamps into real dates and times. The file used 5 different formats, and dates like 03/04 were read as month first.",
     "We need real dates to chart sales by day and to match with weather data.",
     nonstd_ts),
    ("transactions",
     "Kept rows with no timestamp and marked them with has_timestamp = False.",
     "These are still real sales. We just cannot say when they happened, so we do not delete them.",
     int((~tx["has_timestamp"]).sum())),
    ("transactions",
     "Treated negative quantities as returns. We kept them, marked them with is_return, and made their revenue negative.",
     "A return takes money back out of sales. The daily sales summary also has negative revenue, so this matches it.",
     int((pd.to_numeric(raw_tx["quantity"], errors="coerce") < 0).sum())),
    ("transactions",
     "Matched every SKU to one product name (canonical_product).",
     "The same product is sold under several SKUs and spellings. We need to count them as one product for Q2.",
     int(tx["canonical_product"].notna().sum())),
    ("transactions",
     "Deleted exact duplicate rows (same transaction ID and everything else the same).",
     "The same sale was recorded twice, which makes sales look higher than they were.",
     30),
    ("transactions",
     "Deleted rows that are identical except for the transaction ID (same store, time, SKU, quantity, and price). Only done when the row has a timestamp.",
     "These look like the same sale loaded twice. But two real customers could buy the same thing in the same minute, so we should double check this one.",
     30),
    ("stores",
     "Fixed store IDs, set state to FL, turned latitude and longitude into numbers, and kept the most complete row when a store appeared twice.",
     "The duplicate stores were the same store written with different ID formats.",
     len(raw_st) - len(tables["stores"])),
    ("stores",
     "Added a timezone for each store. FL-710 is Central time and all the others are Eastern.",
     "We need the right timezone so store times line up with the weather times.",
     len(tables["stores"])),
    ("distribution_centers",
     "Removed the '~' from surge_capacity and added surge_capacity_is_approx to remember it was an estimate.",
     "We need a number to do math, but we should not forget the value was only approximate.",
     int(raw_dc["surge_capacity"].str.contains("~", na=False).sum())),
    ("product_catalog",
     "Left missing unit_cost values empty. We also decided not to use supplier or package_size.",
     "We do not want to guess costs. Supplier looks wrong in places (for example, Pop-Tarts from a Roku company).",
     int(tables["catalog"]["unit_cost"].isna().sum())),
    ("daily_sales_summary",
     "Kept days with negative revenue and marked them with has_net_return_day.",
     "These are days when more was returned than sold. They are real, not mistakes.",
     int(tables["daily_sales"]["has_net_return_day"].sum())),
    ("inventory_levels",
     "Kept rows for SKUs that are not in the product catalog and marked them with sku_in_catalog = False.",
     "Deleting them would hide real inventory. We can decide later whether to use them.",
     int((~tables["inventory"]["sku_in_catalog"]).sum())),
    ("inventory_levels",
     "Left empty units_on_order values empty instead of changing them to 0.",
     "Empty means we do not know. 0 would wrongly say nothing was ordered.",
     int(tables["inventory"]["units_on_order"].isna().sum())),
]

decision_log = pd.DataFrame(log, columns=["table", "decision", "why", "rows_touched"])
decision_log["team_disagreement"] = "none noted yet - confirm with group"

pd.set_option("display.max_colwidth", None)
display(decision_log)
print("decisions logged:", len(decision_log))

,table,decision,why,rows_touched,team_disagreement
0,transactions,"Made all the different ways of writing 'missing' (blank, NULL, N/A, NA, NaN, None, -) into one real missing value.","Python only treats real missing values as missing. Without this, 'N/A' would be counted as actual data.",886,none noted yet - confirm with group
1,transactions,Made every store ID look the same: FL-###.,"The same store was written many ways (FL-287, fl_287, FL287). They have to match to join to the store file.",3610,none noted yet - confirm with group
2,transactions,Changed category names to the same 10 names used in the daily sales summary.,So our transaction totals can be compared to the summary table.,18760,none noted yet - confirm with group
3,transactions,Removed extra spaces and made every SKU uppercase.,'pt-12' and 'PT-12 ' would otherwise look like two different products.,0,none noted yet - confirm with group
4,transactions,"Turned unit_price from text into a number by removing $, commas, and spaces.","Prices came in as text, and some had a $, so we could not do math on them.",4332,none noted yet - confirm with group
5,transactions,"Turned timestamps into real dates and times. The file used 5 different formats, and dates like 03/04 were read as month first.",We need real dates to chart sales by day and to match with weather data.,13207,none noted yet - confirm with group
6,transactions,Kept rows with no timestamp and marked them with has_timestamp = False.,"These are still real sales. We just cannot say when they happened, so we do not delete them.",884,none noted yet - confirm with group
7,transactions,"Treated negative quantities as returns. We kept them, marked them with is_return, and made their revenue negative.","A return takes money back out of sales. The daily sales summary also has negative revenue, so this matches it.",130,none noted yet - confirm with group
8,transactions,Matched every SKU to one product name (canonical_product).,The same product is sold under several SKUs and spellings. We need to count them as one product for Q2.,25502,none noted yet - confirm with group
9,transactions,Deleted exact duplicate rows (same transaction ID and everything else the same).,"The same sale was recorded twice, which makes sales look higher than they were.",30,none noted yet - confirm with group


decisions logged: 18


### 3 — SKU consolidation progress

**TODO:** how many distinct SKUs did you start with, how many canonical products do you have now, and what is your largest consolidated group? Show the mapping for at least one product family. Pop-Tarts is a good one to start with.

You do not need this finished — you need it started and defensible.

In [6]:
tx = tables["transactions"]

print("Distinct original SKUs:", tx["sku"].nunique())
print("Distinct canonical products:",
      tx["canonical_product"].nunique())
print("Unmapped transaction SKUs:",
      tx.loc[tx["canonical_product"].isna(), "sku"].nunique())

sku_groups = (
    tx.dropna(subset=["canonical_product"])
      .groupby("canonical_product")["sku"]
      .nunique()
      .sort_values(ascending=False)
)
print("\nLargest consolidated product group:")
print(sku_groups.head(1))

print("\nPop-Tarts SKU mapping:")
print(pd.DataFrame([
    {"sku": sku, "canonical_product": product}
    for sku, product in SKU_TO_PRODUCT.items()
    if product == "Pop-Tarts Strawberry"
]))

Distinct original SKUs: 45
Distinct canonical products: 20
Unmapped transaction SKUs: 0

Largest consolidated product group:
canonical_product
Pop-Tarts Strawberry    4
Name: sku, dtype: int64

Pop-Tarts SKU mapping:
             sku     canonical_product
0          PT-12  Pop-Tarts Strawberry
1     SKU#459812  Pop-Tarts Strawberry
2  POPTART-STRAW  Pop-Tarts Strawberry
3  PT_STRAWBERRY  Pop-Tarts Strawberry


### 4 — One working weather API call

**TODO:** show the call and the parsed DataFrame. It must include the retry logic and a `source` column. Say which timezone the weather timestamps are in and how you will line them up with the sales timestamps. If it is not working, paste the actual error and what you have tried.

### 5 — One chart

**TODO:** daily order volume across your date range. It does not need to be presentation quality yet, but it does need labeled axes and a title.

Look at it before you submit. Does the shape make sense? An early chart is the fastest way to catch a cleaning bug — if your daily volume has a day at zero or a day at ten times everything else, that is usually your pipeline, not the hurricane.

### 6 — Self-check

Fill in what you have and leave the rest. **The required items are the team setup and the charter.** Everything else is progress we want to see, not a pass or fail. The checks are shape checks; they cannot verify your analysis is right, only that you have gotten far enough to have one.

In [7]:
required = {
    'team_setup_complete': check_team(team_setup, quiet=True),   # from the team setup section
    'charter_committed': True,         # TODO: True once CHARTER.md shows in the git log above
}

progress = {

    'rows_after_cleaning': len(tables["transactions"]),
    'decisions_logged': len(decision_log),
    'canonical_products': tables["transactions"]["canonical_product"].nunique(),
    'weather_api_rows': None,  # Replace after a successful API call
    'notebook_runs_clean': False,  # Change after Restart and Run All succeeds

}

def is_done(value):
    return not (value is None or value is False
                or (isinstance(value, str) and value.startswith('TODO')))

def check(required, progress):
    print('REQUIRED')
    for label, value in required.items():
        print('  PASS' if is_done(value) else '  FAIL', '-', label, '=', value)
    print('PROGRESS (fill in what you have)')
    for label, value in progress.items():
        print('  done' if is_done(value) else '  not yet', '-', label, '=', value)
    ok = all(is_done(v) for v in required.values())
    print()
    print('required items complete:', ok)
    print('progress items done:', sum(is_done(v) for v in progress.values()), 'of', len(progress))
    return ok

check(required, progress)

REQUIRED
  PASS - team_setup_complete = True
  PASS - charter_committed = True
PROGRESS (fill in what you have)
  done - rows_after_cleaning = 25502
  done - decisions_logged = 18
  done - canonical_products = 20
  not yet - weather_api_rows = None
  not yet - notebook_runs_clean = False

required items complete: True
progress items done: 3 of 5


True

### 7 — Risk and plan

Two questions, answered concretely.

**a)** What is the single most likely thing to go wrong in the next two weeks? Name a specific technical or team problem, not "running out of time."

**b)** What are you doing about it this week? Name who is doing it and by when.

**Biggest risk:** _..._

**Plan, with an owner and a date:** _..._

### 8 — Where each question stands

**TODO:** mark each of the five questions. These owners must match the question owners table in your charter. Be honest — this is the section I read first.

| Question | Status | Who owns it |
|---|---|---|
| Q1 demand surge | not started / in progress / draft answer | _..._ |
| Q2 SKU consolidation | _..._ | _..._ |
| Q3 weather correlation |  | _..._ |
| Q4 store and geography | _..._ | _..._ |
| Q5 signal or artifact | _..._ | _..._ |

At the end of week 1, "not started" on Q4 and Q5 is expected. "Not started" on all five means come find me on Monday.